In [1]:
import pandas as pd
import numpy as np
from glob import glob

In [2]:
filepath_pattern = 'Data/AQI_Pollutant/*.csv'
file_list = sorted(glob(filepath_pattern)) 
dataframes = []
cols_to_keep = ['Date', 'Daily Mean PM2.5 Concentration', 'Daily AQI Value', 'Site Latitude', 'Site Longitude', 'Local Site Name']


In [3]:
for file in file_list:
    df = pd.read_csv(file)
    df = df.loc[:, df.nunique(dropna=True) > 1]
    df = df[cols_to_keep]
    df['Date'] = pd.to_datetime(df['Date'])
    dataframes.append(df)

merged_df = pd.concat(dataframes, ignore_index=True)
temp = merged_df.groupby(['Site Latitude', 'Site Longitude'])[['Local Site Name']].count()
coords_to_drop = temp[temp['Local Site Name'] == 0].index

merged_df = merged_df[~merged_df.set_index(['Site Latitude', 'Site Longitude']).index.isin(coords_to_drop)]
duplicates = merged_df.duplicated(keep='first')
merged_df = merged_df[~duplicates]
merged_df['Date'] = pd.to_datetime(merged_df['Date'])
merged_df.head();

In [7]:
merged_df = (
    merged_df.sort_values(by='Date')
    .groupby(['Site Latitude', 'Site Longitude', 'Date'], as_index=False)
    .first()
)

merged_df.head() 
#printing this for verification
san_jose_df = merged_df[merged_df['Local Site Name'] == 'San Jose - Jackson']
san_jose_df.head()

,Site Latitude,Site Longitude,Date,Daily Mean PM2.5 Concentration,Daily AQI Value,Local Site Name
293595,37.348497,-121.894898,2014-01-01,60.4,154,San Jose - Jackson
293596,37.348497,-121.894898,2014-01-02,33.0,96,San Jose - Jackson
293597,37.348497,-121.894898,2014-01-03,32.6,95,San Jose - Jackson
293598,37.348497,-121.894898,2014-01-04,26.4,83,San Jose - Jackson
293599,37.348497,-121.894898,2014-01-05,18.1,68,San Jose - Jackson


In [ ]:
grouped = (
    merged_df.groupby(['Site Latitude', 'Site Longitude'])
    .size()
    .reset_index(name='Count')
    .sort_values('Count', ascending=False)
)
top_20_coords = grouped.head(20)[['Site Latitude', 'Site Longitude']]


merged_df_top_20 = merged_df.merge(top_20_coords, on=['Site Latitude', 'Site Longitude'])
merged_df_top_20 = merged_df_top_20.sort_values(by=['Local Site Name', 'Date'], ascending=True).reset_index(drop=True)
merged_df_top_20.head()

In [ ]:
merged_df_top_20 = merged_df_top_20.rename(columns={
    'Site Latitude': 'Latitude',
    'Site Longitude': 'Longitude',
    'Daily Mean PM2.5 Concentration': 'PM2.5',
    'Daily AQI Value': 'AQI',
    'Local Site Name': 'Name'
})

merged_df_top_20.head()

In [ ]:
merged_df_top_20.set_index('Date', inplace=True)
merged_df_top_20.to_csv('Data/top_20_cities_pollution_data.csv', index='Date')

In [ ]:
city_locations = merged_df_top_20.groupby(['Latitude', 'Longitude', 'Name']).size().reset_index(name='Count')
city_locations[['Latitude', 'Longitude', 'Name']]